# DCA Buy-the-Bear — Scaled Buy (2% of Portfolio)

Instead of fixed $10, buy 2% of current portfolio value every red monthly candle.
Sell 70% at every new ATH. No separate reserve reinvestment — the scaled buy naturally compounds.

In [ ]:
import pandas as pd
import requests
import time
import json
import numpy as np
from datetime import datetime, timezone

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: '%.6f' % x)

In [ ]:
def fetch_monthly_klines(symbol='BTCUSDT', limit=500):
    url = 'https://api.binance.com/api/v3/klines'
    all_klines = []
    start_time = 0
    while True:
        params = {'symbol': symbol, 'interval': '1M', 'startTime': start_time, 'limit': limit}
        resp = requests.get(url, params=params)
        data = resp.json()
        if not data or len(data) == 0:
            break
        all_klines.extend(data)
        print(f'Fetched {len(data)} months, up to {datetime.fromtimestamp(data[-1][0]/1000, tz=timezone.utc).strftime("%Y-%m")}')
        if len(data) < limit:
            break
        start_time = data[-1][0] + 1
        time.sleep(0.1)
    return all_klines

klines = fetch_monthly_klines()

In [ ]:
columns = ['timestamp', 'open', 'high', 'low', 'close', 'volume',
           'close_time', 'quote_vol', 'trades', 'taker_buy_base', 'taker_buy_quote', 'ignore']
df = pd.DataFrame(klines, columns=columns)
df['date'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
for c in ['open', 'high', 'low', 'close', 'volume']:
    df[c] = df[c].astype(float)
df = df[['date', 'open', 'high', 'low', 'close', 'volume']].copy()
df = df.sort_values('date').reset_index(drop=True)
print(f'{df["date"].min().strftime("%Y-%m")} → {df["date"].max().strftime("%Y-%m")} ({len(df)} months)')

In [ ]:
BUY_PCT = 2.0  # buy this % of portfolio value each red month

btc_held = 0.0
usdt_reserve = 0.0
total_invested = 0.0
highest_close = 0.0

records = []

for i, row in df.iterrows():
    close = row['close']
    month = row['date']
    action = 'nothing'
    portfolio_value = btc_held * close + usdt_reserve

    # Entry: buy BUY_PCT% of portfolio value on red monthly candle
    if close < row['open']:
        buy_amount = max(portfolio_value * (BUY_PCT / 100.0), 10.0)  # minimum $10
        btc_bought = buy_amount / close
        btc_held += btc_bought
        total_invested += buy_amount
        action = f'buy ${buy_amount:.1f} ({BUY_PCT}% of ${portfolio_value:.0f}) @ {close:.0f}'

    prev_highest = highest_close
    if close > highest_close:
        highest_close = close

    if btc_held > 0.000001 and close > prev_highest:
        sell_btc = btc_held * 0.7
        sell_usdt = sell_btc * close
        btc_held -= sell_btc
        usdt_reserve += sell_usdt
        act = f'sell 70% @ {close:.0f} (-{sell_btc:.6f} BTC, +{sell_usdt:.1f} USDT)'
        action = f'{action} | {act}' if 'buy' in action else act

    portfolio_value = btc_held * close + usdt_reserve
    btc_value = btc_held * close

    records.append({
        'date': month, 'close': close, 'action': action,
        'btc_held': btc_held, 'btc_value': btc_value,
        'usdt_reserve': usdt_reserve, 'total_invested': total_invested,
        'portfolio_value': portfolio_value,
    })

results = pd.DataFrame(records)
print(f'Total months: {len(results)}')
print(f'Trades with action: {len(results[results["action"] != "nothing"])}')

In [ ]:
# Metrics

final = results.iloc[-1]
last_close = final['close']
total_pnl = final['portfolio_value'] - final['total_invested']
ret_pct = (final['portfolio_value'] / final['total_invested'] - 1) * 100

eq = results['portfolio_value']
running_max = eq.cummax()
dd_raw = (running_max - eq) / running_max
max_dd = dd_raw.max() * 100

monthly_returns = eq.pct_change().dropna()
monthly_returns = monthly_returns[monthly_returns.index >= 12]
sharpe = (monthly_returns.mean() / monthly_returns.std()) * np.sqrt(12) if monthly_returns.std() > 0 else 0.0

pos_sum = monthly_returns[monthly_returns > 0].sum()
neg_sum = monthly_returns[monthly_returns < 0].abs().sum()
pf = pos_sum / neg_sum if neg_sum > 0 else float('inf')

annualized_ret = (1 + ret_pct / 100) ** (12 / len(df)) - 1
calmar = annualized_ret / (max_dd / 100) if max_dd > 0 else 0

buys = [r for r in records if 'buy' in r['action']]
sells = [r for r in records if 'sell' in r['action']]

print('='*65)
print('PORTFOLIO SUMMARY — Scaled Buy (2% of Portfolio)')
print('='*65)
print(f'Buy size:              {BUY_PCT}% of portfolio value (min $10)')
print(f'Total invested:        {final["total_invested"]:.2f} USDT')
print(f'BTC held:              {final["btc_held"]:.8f} BTC')
print(f'BTC value:             {final["btc_value"]:.2f} USDT (at {last_close:.2f})')
print(f'USDT reserve:          {final["usdt_reserve"]:.2f} USDT')
print(f'Portfolio value:       {final["portfolio_value"]:.2f} USDT')
print(f'Total P&L:             {total_pnl:.2f} USDT')
print(f'Return:                {ret_pct:.2f}%')
print(f'Max drawdown:          {max_dd:.2f}%')
print(f'Sharpe ratio:          {sharpe:.2f}')
print(f'Profit factor:         {pf:.2f}')
print(f'Calmar ratio:          {calmar:.2f}')
print(f'Months:                {len(results)}')
print(f'Buy months:            {len(buys)}')
print(f'Sell months:           {len(sells)}')
print()

# Compare with fixed $10 final version
print('='*65)
print('COMPARISON: Fixed $10+Progressive vs Scaled 2%')
print('='*65)
print(f'{"":>30s} {"Fixed $10":>12s} {"Scaled 2%":>12s}')
print(f'{"Portfolio value":>30s} {"$2,396":>12s} {f"${final["portfolio_value"]:.0f}":>12s}')
print(f'{"Total invested":>30s} {"$510":>12s} {f"${final["total_invested"]:.0f}":>12s}')
print(f'{"Return":>30s} {"369.9%":>12s} {f"{ret_pct:.1f}%":>12s}')
print(f'{"Max DD":>30s} {"24.5%":>12s} {f"{max_dd:.1f}%":>12s}')
print(f'{"Sharpe":>30s} {"1.30":>12s} {f"{sharpe:.2f}":>12s}')
print(f'{"Calmar":>30s} {"0.77":>12s} {f"{calmar:.2f}":>12s}')

In [ ]:
# Optimise: test different buy percentages

def run_test(pct):
    btc_held = 0.0; usdt_reserve = 0.0; total_invested = 0.0; highest_close = 0.0
    records = []
    for i, row in df.iterrows():
        close = row['close']
        portfolio_value = btc_held * close + usdt_reserve
        if close < row['open']:
            buy_amount = max(portfolio_value * (pct / 100.0), 10.0)
            btc_bought = buy_amount / close
            btc_held += btc_bought
            total_invested += buy_amount
        prev_highest = highest_close
        if close > highest_close:
            highest_close = close
        if btc_held > 0.000001 and close > prev_highest:
            sell_btc = btc_held * 0.7
            usdt_reserve += sell_btc * close
            btc_held -= sell_btc
        records.append(btc_held * close + usdt_reserve)
    final_val = records[-1]
    ret = (final_val / total_invested - 1) * 100
    eq = pd.Series(records)
    running_max = eq.cummax()
    dd = ((running_max - eq) / running_max).max() * 100
    mr = eq.pct_change().dropna()
    mr = mr[mr.index >= 12]
    sharpe = (mr.mean() / mr.std()) * np.sqrt(12) if mr.std() > 0 else 0
    ps = mr[mr > 0].sum(); ns = mr[mr < 0].abs().sum()
    pf = ps / ns if ns > 0 else float('inf')
    ann = (1 + ret / 100) ** (12 / len(df)) - 1
    calmar = ann / (dd / 100) if dd > 0 else 0
    return {'pct': pct, 'final_val': final_val, 'invested': total_invested,
            'return_pct': ret, 'max_dd': dd, 'sharpe': sharpe, 'calmar': calmar, 'pf': pf}

print("Testing buy percentages from 0.5% to 10%...")
pcts = [0.5, 1, 1.5, 2, 2.5, 3, 4, 5, 6, 7, 8, 9, 10]
opt_list = [run_test(p) for p in pcts]
opt_df = pd.DataFrame(opt_list)
print()
print(f"{'Buy%':>6s} {'Invested':>10s} {'Final':>10s} {'Return%':>8s} {'DD%':>6s} {'Sharpe':>7s} {'Calmar':>7s}")
print('-'*60)
for _, r in opt_df.iterrows():
    print(f"{r['pct']:>5.1f}% ${r['invested']:>8.0f} ${r['final_val']:>8.0f} {r['return_pct']:>7.1f}% {r['max_dd']:>5.1f}% {r['sharpe']:>7.2f} {r['calmar']:>7.2f}")

In [ ]:
# Chart: portfolio growth + drawdown + BTC value breakdown

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, axes = plt.subplots(3, 1, figsize=(14, 9), gridspec_kw={'height_ratios': [3, 1, 1]})

# Top: portfolio
ax = axes[0]
ax.fill_between(results['date'], results['total_invested'], results['portfolio_value'],
                where=results['portfolio_value'] >= results['total_invested'],
                color='green', alpha=0.12, label='Profit')
ax.fill_between(results['date'], results['portfolio_value'], results['total_invested'],
                where=results['portfolio_value'] < results['total_invested'],
                color='red', alpha=0.12, label='Loss')
ax.plot(results['date'], results['total_invested'], color='gray', linewidth=1.5, linestyle='--', label='Total Invested')
ax.plot(results['date'], results['portfolio_value'], color='#2563eb', linewidth=2, label='Portfolio Value')

buys_df = results[results['action'].str.contains('buy', na=False)]
sells_df = results[results['action'].str.contains('sell', na=False)]
ax.scatter(buys_df['date'], buys_df['portfolio_value'], color='#16a34a', s=20, marker='^', zorder=5, alpha=0.6)
ax.scatter(sells_df['date'], sells_df['portfolio_value'], color='#dc2626', s=50, marker='v', zorder=5, label='Sell 70%')

ax.set_title(f'DCA Buy-the-Bear — Scaled Buy ({BUY_PCT}% of Portfolio per Red Month)', fontsize=13, fontweight='bold')
ax.set_ylabel('USDT')
ax.legend(loc='upper left', fontsize=8, ncol=3)
ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Middle: drawdown
ax = axes[1]
ax.fill_between(results['date'], 0, dd_raw * 100, color='#dc2626', alpha=0.35)
ax.plot(results['date'], dd_raw * 100, color='#dc2626', linewidth=0.8)
ax.axhline(y=-max_dd, color='#991b1b', linestyle=':', linewidth=1.2, alpha=0.8, label=f'Max DD: {max_dd:.1f}%')
ax.set_ylabel('Drawdown (%)')
ax.legend(loc='lower left', fontsize=8)
ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Bottom: BTC value + USDT reserve stacked
ax = axes[2]
ax.fill_between(results['date'], 0, results['btc_value'], color='#f59e0b', alpha=0.6, label='BTC Value')
ax.fill_between(results['date'], results['btc_value'], results['btc_value'] + results['usdt_reserve'],
                color='#3b82f6', alpha=0.4, label='USDT Reserve')
ax.set_ylabel('USDT')
ax.set_xlabel('Date')
ax.legend(loc='upper left', fontsize=8, ncol=2)
ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('dca-bt-buy-bot-scaled-2pct.png', dpi=150, bbox_inches='tight')
print('Chart saved as dca-bt-buy-bot-scaled-2pct.png')
plt.show()